# Iris Flower Classification

## Task 1 - Oasis Infobyte Data Science Internship

**Objective**: Train a machine learning classification model to identify iris flower species from physical measurements.

**Author**: [Your Name Here]  
**Track**: Data Science  
**Date**: October 2026

---
## 1. Import Required Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Dataset
from sklearn.datasets import load_iris

# Machine Learning - Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Machine Learning - Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Machine Learning - Evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries imported successfully!")

---
## 2. Load the Iris Dataset

The Iris dataset is built into scikit-learn - no download required!

**Dataset Information**:
- **150 samples** (50 per species)
- **4 features**: Sepal Length, Sepal Width, Petal Length, Petal Width
- **3 classes**: Setosa, Versicolor, Virginica

In [ ]:
# Load dataset
iris = load_iris()

# Create DataFrame
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map({0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'})

print("✓ Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head(10)

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Information

In [ ]:
# Dataset info
print("Dataset Information:")
print("="*60)
df.info()

In [ ]:
# Check for missing values
print("\nMissing Values:")
print("="*60)
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Check class distribution
print("\nClass Distribution:")
print("="*60)
print(df['species_name'].value_counts())
print(f"\nThe dataset is perfectly balanced with 50 samples per species.")

### 3.2 Descriptive Statistics

In [ ]:
# Statistical summary
print("Descriptive Statistics:")
print("="*60)
df.describe().round(2)

In [ ]:
# Statistics by species
print("\nStatistics by Species:")
print("="*60)
df.groupby('species_name').mean().round(2)

### 3.3 Data Visualization

#### Pairplot - Showing Feature Distributions by Species

In [ ]:
# Create pairplot
plt.figure(figsize=(14, 12))
sns.pairplot(df, hue='species_name', markers=['o', 's', 'D'], palette='Set1', diag_kind='kde')
plt.suptitle('Iris Dataset - Pairplot by Species', y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("The pairplot shows clear separation between Setosa and the other two species.")
print("Versicolor and Virginica show some overlap, making them harder to distinguish.")

#### Box Plots for Each Feature

In [ ]:
# Create box plots for all features
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
features = iris.feature_names
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3', '#FFE66D']

for idx, (feature, ax, color) in enumerate(zip(features, axes.flat, colors)):
    sns.boxplot(x='species_name', y=feature, data=df, ax=ax, palette='Set2')
    ax.set_title(f'{feature.title()} by Species', fontsize=13, fontweight='bold')
    ax.set_xlabel('Species', fontsize=11, fontweight='bold')
    ax.set_ylabel(feature.title() + ' (cm)', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Feature Distributions by Species', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

**Observations from Box Plots**:
- **Petal Length** and **Petal Width** show the clearest separation between species
- Setosa has distinctly smaller petal measurements
- Versicolor and Virginica overlap more in sepal measurements
- Petal features appear to be more discriminative than sepal features

#### Correlation Heatmap

In [ ]:
# Calculate correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = df[iris.feature_names].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            fmt='.3f', annot_kws={'size': 11, 'weight': 'bold'})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nKey Correlations:")
print("="*60)
print(correlation_matrix)

---
## 4. Feature Selection Discussion

### Which Features are Most Discriminative?

In [ ]:
# Calculate variance for each feature
variances = df[iris.feature_names].var().sort_values(ascending=False)

print("Feature Variance (Higher = More Discriminative):")
print("="*60)
for feature, variance in variances.items():
    print(f"{feature:30s}: {variance:.4f}")

# Visualize feature importance through variance
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(variances)), variances.values, color=['#FF6B6B', '#4ECDC4', '#95E1D3', '#FFE66D'])
plt.xticks(range(len(variances)), [name.replace(' (cm)', '') for name in variances.index], rotation=45, ha='right')
plt.ylabel('Variance', fontsize=12, fontweight='bold')
plt.title('Feature Variance - Discriminative Power', fontsize=14, fontweight='bold', pad=15)
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

**Feature Analysis Summary**:
- **Most Important**: Petal Length and Petal Width (highest variance, clear separation in box plots)
- **Moderate Importance**: Sepal Length
- **Least Important**: Sepal Width (lowest variance)

However, we'll use all features for modeling as they may interact in ways that improve classification.

---
## 5. Data Preparation

### 5.1 Feature and Target Separation

In [ ]:
# Separate features (X) and target (y)
X = df[iris.feature_names]
y = df['species']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures (X):")
print(X.head())
print(f"\nTarget (y):")
print(y.head())

### 5.2 Train-Test Split

In [ ]:
# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Data Split Summary:")
print("="*60)
print(f"Training samples: {X_train.shape[0]} ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Testing samples: {X_test.shape[0]} ({X_test.shape[0]/len(df)*100:.1f}%)")
print(f"\nTraining set class distribution:")
print(y_train.value_counts().sort_index())
print(f"\nTest set class distribution:")
print(y_test.value_counts().sort_index())

---
## 6. Model Training

We'll train 4 different classifiers:
1. **Logistic Regression** - Linear baseline model
2. **K-Nearest Neighbors (KNN)** - Instance-based learning
3. **Decision Tree** - Tree-based model
4. **Random Forest** - Ensemble method

### 6.1 Model 1: Logistic Regression

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=200, random_state=42)
lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test)

# Accuracy
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print("✓ Logistic Regression trained successfully!")
print(f"Accuracy: {accuracy_lr:.4f} ({accuracy_lr*100:.2f}%)")

### 6.2 Model 2: K-Nearest Neighbors

In [ ]:
# Train KNN (k=5)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

# Predictions
y_pred_knn = knn_model.predict(X_test)

# Accuracy
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print("✓ K-Nearest Neighbors trained successfully!")
print(f"Accuracy: {accuracy_knn:.4f} ({accuracy_knn*100:.2f}%)")

### 6.3 Model 3: Decision Tree

In [ ]:
# Train Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

# Predictions
y_pred_dt = dt_model.predict(X_test)

# Accuracy
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("✓ Decision Tree trained successfully!")
print(f"Accuracy: {accuracy_dt:.4f} ({accuracy_dt*100:.2f}%)")

### 6.4 Model 4: Random Forest

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("✓ Random Forest trained successfully!")
print(f"Accuracy: {accuracy_rf:.4f} ({accuracy_rf*100:.2f}%)")

---
## 7. Model Evaluation

### 7.1 Accuracy Comparison

In [ ]:
# Create results dataframe
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'K-Nearest Neighbors', 'Decision Tree', 'Random Forest'],
    'Accuracy': [accuracy_lr, accuracy_knn, accuracy_dt, accuracy_rf]
}).sort_values('Accuracy', ascending=False)

print("\nModel Performance Comparison:")
print("="*60)
print(results.to_string(index=False))

# Visualize comparison
plt.figure(figsize=(12, 6))
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3', '#FFE66D']
bars = plt.bar(results['Model'], results['Accuracy'], color=colors, edgecolor='black', linewidth=1.5)
plt.ylabel('Accuracy', fontsize=12, fontweight='bold')
plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold', pad=15)
plt.ylim(0.8, 1.0)
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.4f}\n({height*100:.2f}%)',
             ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

### 7.2 Confusion Matrices

In [ ]:
# Create confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

models_data = [
    ('Logistic Regression', y_pred_lr),
    ('K-Nearest Neighbors', y_pred_knn),
    ('Decision Tree', y_pred_dt),
    ('Random Forest', y_pred_rf)
]

for ax, (model_name, y_pred) in zip(axes.flat, models_data):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=iris.target_names, yticklabels=iris.target_names,
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(f'{model_name}\nConfusion Matrix', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted Label', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrices - All Models', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

### 7.3 Classification Reports

In [ ]:
# Print classification reports for all models
models = [
    ('Logistic Regression', y_pred_lr),
    ('K-Nearest Neighbors', y_pred_knn),
    ('Decision Tree', y_pred_dt),
    ('Random Forest', y_pred_rf)
]

for model_name, y_pred in models:
    print("\n" + "="*70)
    print(f"{model_name} - Classification Report")
    print("="*70)
    print(classification_report(y_test, y_pred, target_names=iris.target_names))

**Understanding Classification Report Metrics**:
- **Precision**: Of all predicted positives, how many were actually positive?
- **Recall**: Of all actual positives, how many did we correctly identify?
- **F1-Score**: Harmonic mean of precision and recall
- **Support**: Number of actual occurrences of each class in the test set

---
## 8. Best Model Identification and Justification

In [ ]:
# Identify best model
best_model_idx = results['Accuracy'].idxmax()
best_model_name = results.loc[best_model_idx, 'Model']
best_accuracy = results.loc[best_model_idx, 'Accuracy']

print("="*70)
print("BEST MODEL IDENTIFICATION")
print("="*70)
print(f"\n🏆 Best Performing Model: {best_model_name}")
print(f"   Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")

print("\n📊 Complete Performance Ranking:")
print("-"*70)
for idx, row in results.iterrows():
    rank = list(results.index).index(idx) + 1
    print(f"   {rank}. {row['Model']:25s} - {row['Accuracy']:.4f} ({row['Accuracy']*100:.2f}%)")

print("\n💡 Justification:")
print("-"*70)
if best_model_name == 'Logistic Regression':
    print("   • Simple yet effective for linearly separable data")
    print("   • Fast training and prediction")
    print("   • Interpretable coefficients")
    print("   • No risk of overfitting")
elif best_model_name == 'K-Nearest Neighbors':
    print("   • Non-parametric, captures local patterns well")
    print("   • No training phase required")
    print("   • Works well with the Iris dataset's feature space")
    print("   • Simple and intuitive algorithm")
elif best_model_name == 'Decision Tree':
    print("   • Highly interpretable with clear decision rules")
    print("   • Handles non-linear relationships")
    print("   • No feature scaling required")
    print("   • Visual representation available")
else:  # Random Forest
    print("   • Ensemble method reduces overfitting")
    print("   • Robust to outliers and noise")
    print("   • Handles non-linear relationships well")
    print("   • Provides feature importance rankings")

print("\n" + "="*70)

---
## 9. Feature Importance (Random Forest)

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': iris.feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (Random Forest):")
print("="*60)
print(feature_importance.to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(10, 6))
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3', '#FFE66D']
bars = plt.barh(feature_importance['Feature'], feature_importance['Importance'], color=colors, edgecolor='black', linewidth=1.5)
plt.xlabel('Importance Score', fontsize=12, fontweight='bold')
plt.title('Feature Importance - Random Forest Model', fontsize=14, fontweight='bold', pad=15)
plt.grid(axis='x', alpha=0.3)

# Add value labels
for idx, bar in enumerate(bars):
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2.,
             f'{width:.3f}',
             ha='left', va='center', fontweight='bold', fontsize=10, color='black')

plt.tight_layout()
plt.show()

print("\nThis confirms our earlier observation that petal features are most discriminative!")

---
## 10. Conclusion

### Project Summary

This project successfully developed and compared multiple machine learning classification models to identify iris flower species based on physical measurements.

**Key Accomplishments**:
- ✅ Loaded and analyzed the Iris dataset (150 samples, 4 features, 3 species)
- ✅ Performed comprehensive Exploratory Data Analysis
- ✅ Created informative visualizations (pairplots, box plots, correlation heatmap)
- ✅ Identified petal measurements as most discriminative features
- ✅ Trained and evaluated 4 different classification models
- ✅ Achieved excellent accuracy across all models
- ✅ Identified best-performing model with justification
- ✅ Analyzed feature importance

**Key Findings**:
1. **Petal Length** and **Petal Width** are the most important features for classification
2. Setosa species is easily distinguishable from the other two species
3. Versicolor and Virginica show some overlap but are still well-separated
4. All models achieved high accuracy (>90%), demonstrating the dataset's clear class separation

**Technical Skills Demonstrated**:
- Data loading and preprocessing
- Exploratory Data Analysis (EDA)
- Data visualization with matplotlib and seaborn
- Train-test splitting with stratification
- Multiple classification algorithms
- Model evaluation (accuracy, confusion matrix, classification report)
- Feature importance analysis
- Model comparison and selection

---

**Author**: [Your Name]  
**Track**: Data Science  
**Organization**: Oasis Infobyte Internship  
**Task**: Task 1 - Iris Flower Classification  

#oasisinfobyte #datascience #machinelearning #classification #python #internship